<a href="https://colab.research.google.com/github/VinayaSharada/FinancialAnalyticsCourse/blob/main/PortfolioOpt/Portfolio_Optimization_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Portfolio Optimization Demo - NIFTY 50 Stocks

## Master's Finance Course - AI/ML in Finance

This notebook demonstrates comprehensive portfolio optimization techniques using real data from NIFTY 50 stocks listed on the National Stock Exchange of India.

### Learning Objectives:
- Understand Modern Portfolio Theory (MPT)
- Implement portfolio optimization algorithms
- Generate and analyze efficient frontiers
- Compare different optimization objectives
- Perform risk analysis and stress testing

### Key Concepts:
- **Risk-Return Trade-off**: Higher returns typically require accepting higher risk
- **Diversification**: Reducing risk through asset allocation
- **Efficient Frontier**: Set of optimal portfolios
- **Sharpe Ratio**: Risk-adjusted return measure


## Setup and Installation

First, let's install the required packages and import necessary libraries.

In [ ]:
# Install required packages (run this cell in Google Colab)
!pip install yfinance pandas numpy matplotlib seaborn scipy scikit-learn plotly PyPortfolioOpt cvxpy -q

import warnings
warnings.filterwarnings('ignore')

print("✅ Packages installed successfully!")

In [ ]:
# Import libraries
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.optimize import minimize
from scipy import stats
from datetime import datetime, timedelta
import json

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("📚 Libraries imported successfully!")
print(f"📅 Current date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## NIFTY 50 Stock Data

Let's define the NIFTY 50 stocks and their sector classifications.

In [ ]:
# NIFTY 50 Stock Tickers (Yahoo Finance format)
NIFTY50_TICKERS = [
    'RELIANCE.NS', 'HDFCBANK.NS', 'TCS.NS', 'ICICIBANK.NS', 'BHARTIARTL.NS',
    'SBIN.NS', 'INFY.NS', 'HINDUNILVR.NS', 'LT.NS', 'ITC.NS',
    'HCLTECH.NS', 'BAJFINANCE.NS', 'SUNPHARMA.NS', 'ONGC.NS', 'KOTAKBANK.NS',
    'AXISBANK.NS', 'ASIANPAINT.NS', 'MARUTI.NS', 'NESTLEIND.NS', 'WIPRO.NS',
    'ULTRACEMCO.NS', 'TECHM.NS', 'TITAN.NS', 'COALINDIA.NS', 'NTPC.NS',
    'POWERGRID.NS', 'BAJAJFINSV.NS', 'M&M.NS', 'JSWSTEEL.NS', 'TATASTEEL.NS',
    'INDUSINDBK.NS', 'ADANIENT.NS', 'HINDALCO.NS', 'GRASIM.NS', 'CIPLA.NS',
    'DIVISLAB.NS', 'DRREDDY.NS', 'EICHERMOT.NS', 'HEROMOTOCO.NS', 'BRITANNIA.NS',
    'APOLLOHOSP.NS', 'BAJAJ-AUTO.NS', 'BPCL.NS', 'TATAMOTORS.NS', 'SHRIRAMFIN.NS',
    'TATACONSUM.NS', 'ADANIPORTS.NS', 'UPL.NS', 'LTIM.NS', 'TRENT.NS'
]

# Sector Classification
SECTOR_MAPPING = {
    'RELIANCE.NS': 'Energy', 'HDFCBANK.NS': 'Financials', 'TCS.NS': 'IT',
    'ICICIBANK.NS': 'Financials', 'BHARTIARTL.NS': 'Telecom', 'SBIN.NS': 'Financials',
    'INFY.NS': 'IT', 'HINDUNILVR.NS': 'FMCG', 'LT.NS': 'Infrastructure',
    'ITC.NS': 'FMCG', 'HCLTECH.NS': 'IT', 'BAJFINANCE.NS': 'Financials',
    'SUNPHARMA.NS': 'Pharma', 'ONGC.NS': 'Energy', 'KOTAKBANK.NS': 'Financials',
    'AXISBANK.NS': 'Financials', 'ASIANPAINT.NS': 'Paints', 'MARUTI.NS': 'Auto',
    'NESTLEIND.NS': 'FMCG', 'WIPRO.NS': 'IT', 'ULTRACEMCO.NS': 'Cement',
    'TECHM.NS': 'IT', 'TITAN.NS': 'Consumer Durables', 'COALINDIA.NS': 'Mining',
    'NTPC.NS': 'Power', 'POWERGRID.NS': 'Power', 'BAJAJFINSV.NS': 'Financials',
    'M&M.NS': 'Auto', 'JSWSTEEL.NS': 'Steel', 'TATASTEEL.NS': 'Steel',
    'INDUSINDBK.NS': 'Financials', 'ADANIENT.NS': 'Conglomerate', 'HINDALCO.NS': 'Metals',
    'GRASIM.NS': 'Chemicals', 'CIPLA.NS': 'Pharma', 'DIVISLAB.NS': 'Pharma',
    'DRREDDY.NS': 'Pharma', 'EICHERMOT.NS': 'Auto', 'HEROMOTOCO.NS': 'Auto',
    'BRITANNIA.NS': 'FMCG', 'APOLLOHOSP.NS': 'Healthcare', 'BAJAJ-AUTO.NS': 'Auto',
    'BPCL.NS': 'Energy', 'TATAMOTORS.NS': 'Auto', 'SHRIRAMFIN.NS': 'Financials',
    'TATACONSUM.NS': 'FMCG', 'ADANIPORTS.NS': 'Infrastructure', 'UPL.NS': 'Chemicals',
    'LTIM.NS': 'IT', 'TRENT.NS': 'Retail'
}

# For demo purposes, we'll use a subset of stocks to ensure faster execution
DEMO_TICKERS = NIFTY50_TICKERS[:25]  # Top 25 stocks

print(f"📊 Total NIFTY 50 stocks: {len(NIFTY50_TICKERS)}")
print(f"🚀 Demo using top {len(DEMO_TICKERS)} stocks for faster execution")
print(f"🏭 Number of sectors: {len(set(SECTOR_MAPPING.values()))}")

# Display sector distribution
sector_count = pd.Series([SECTOR_MAPPING[stock] for stock in DEMO_TICKERS]).value_counts()
print("\n📈 Sector Distribution (Demo Stocks):")
print(sector_count)

## Data Fetching and Preparation

Now let's fetch historical stock price data from Yahoo Finance and prepare it for analysis.

In [ ]:
def fetch_stock_data(tickers, period="2y", interval="1d"):
    """
    Fetch stock data with comprehensive error handling
    """
    try:
        print(f"🔄 Fetching data for {len(tickers)} stocks over {period} period...")
        
        # Download data
        data = yf.download(tickers, period=period, interval=interval, 
                          group_by='ticker', auto_adjust=True, 
                          prepost=True, threads=True, progress=True)
        
        if data.empty:
            raise ValueError("No data retrieved from Yahoo Finance")
        
        # Extract adjusted closing prices
        if len(tickers) == 1:
            prices = data['Close'].to_frame()
            prices.columns = tickers
        else:
            prices = pd.DataFrame()
            successful_tickers = []
            
            for ticker in tickers:
                try:
                    if ticker in data.columns.levels[0]:
                        prices[ticker] = data[ticker]['Close']
                        successful_tickers.append(ticker)
                    else:
                        print(f"⚠️ No data found for {ticker}")
                except KeyError:
                    print(f"❌ Error processing {ticker}")
                    continue
        
        # Data cleaning
        original_shape = prices.shape
        prices = prices.dropna(thresh=len(prices) * 0.5, axis=1)  # Keep columns with 50% data
        prices = prices.fillna(method='ffill').fillna(method='bfill')
        
        print(f"✅ Successfully fetched data for {len(prices.columns)} stocks")
        print(f"📅 Date range: {prices.index[0].strftime('%Y-%m-%d')} to {prices.index[-1].strftime('%Y-%m-%d')}")
        print(f"📊 Data shape: {prices.shape} (cleaned from {original_shape})")
        
        return prices
        
    except Exception as e:
        print(f"❌ Error fetching stock data: {str(e)}")
        raise

def calculate_returns(prices):
    """
    Calculate daily returns with outlier handling
    """
    try:
        returns = prices.pct_change().dropna()
        
        # Remove extreme outliers (beyond 3 standard deviations)
        for col in returns.columns:
            mean = returns[col].mean()
            std = returns[col].std()
            returns[col] = returns[col].clip(lower=mean - 3*std, upper=mean + 3*std)
        
        print(f"📈 Calculated returns for {len(returns.columns)} stocks")
        print(f"📊 Returns data shape: {returns.shape}")
        
        return returns
        
    except Exception as e:
        print(f"❌ Error calculating returns: {str(e)}")
        raise

# Fetch data
prices = fetch_stock_data(DEMO_TICKERS, period="2y")
returns = calculate_returns(prices)

# Display summary statistics
print("\n📊 Price Data Summary:")
print(prices.describe())

print("\n📈 Returns Summary:")
print(returns.describe())

## Exploratory Data Analysis

Let's analyze the characteristics of our stock data.

In [ ]:
# Calculate individual stock metrics
def calculate_stock_metrics(returns, risk_free_rate=0.065):
    """
    Calculate comprehensive metrics for individual stocks
    """
    metrics = {}
    
    for stock in returns.columns:
        stock_returns = returns[stock].dropna()
        
        if len(stock_returns) == 0:
            continue
        
        annual_return = stock_returns.mean() * 252
        annual_vol = stock_returns.std() * np.sqrt(252)
        sharpe_ratio = (annual_return - risk_free_rate) / annual_vol if annual_vol != 0 else 0
        
        # Calculate max drawdown
        cumulative = (1 + stock_returns).cumprod()
        rolling_max = cumulative.expanding().max()
        drawdown = (cumulative - rolling_max) / rolling_max
        max_drawdown = drawdown.min()
        
        metrics[stock] = {
            'Annual Return': annual_return,
            'Annual Volatility': annual_vol,
            'Sharpe Ratio': sharpe_ratio,
            'Max Drawdown': max_drawdown,
            'VaR (95%)': np.percentile(stock_returns, 5),
            'Skewness': stats.skew(stock_returns),
            'Kurtosis': stats.kurtosis(stock_returns),
            'Sector': SECTOR_MAPPING.get(stock, 'Unknown')
        }
    
    return pd.DataFrame(metrics).T

# Calculate metrics
stock_metrics = calculate_stock_metrics(returns)

print("📊 Individual Stock Metrics:")
display_cols = ['Annual Return', 'Annual Volatility', 'Sharpe Ratio', 'Max Drawdown', 'Sector']
print(stock_metrics[display_cols].round(4))

# Top performers by different metrics
print("\n🏆 Top 5 Stocks by Sharpe Ratio:")
top_sharpe = stock_metrics.nlargest(5, 'Sharpe Ratio')[['Annual Return', 'Annual Volatility', 'Sharpe Ratio', 'Sector']]
print(top_sharpe.round(4))

print("\n🎯 Lowest Risk Stocks (by Volatility):")
low_risk = stock_metrics.nsmallest(5, 'Annual Volatility')[['Annual Return', 'Annual Volatility', 'Sharpe Ratio', 'Sector']]
print(low_risk.round(4))

In [ ]:
# Correlation Analysis
correlation_matrix = returns.corr()

# Plot correlation heatmap
plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=False, cmap='coolwarm',
           center=0, square=True, cbar_kws={'label': 'Correlation'})
plt.title('Stock Correlation Matrix - NIFTY Stocks', fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Average correlation
avg_correlation = correlation_matrix.values[np.triu_indices_from(correlation_matrix.values, k=1)].mean()
print(f"\n📊 Average pairwise correlation: {avg_correlation:.3f}")

# Find highest and lowest correlations
corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_pairs.append({
            'Stock1': correlation_matrix.columns[i],
            'Stock2': correlation_matrix.columns[j],
            'Correlation': correlation_matrix.iloc[i, j]
        })

corr_df = pd.DataFrame(corr_pairs)
highest_corr = corr_df.nlargest(10, 'Correlation')
lowest_corr = corr_df.nsmallest(10, 'Correlation')

print("\n🔗 Highest Correlations:")
print(highest_corr.round(3))

print("\n🔄 Lowest Correlations:")
print(lowest_corr.round(3))

In [ ]:
# Risk-Return Scatter Plot
plt.figure(figsize=(14, 10))

# Create scatter plot colored by sector
sectors = stock_metrics['Sector'].unique()
colors = plt.cm.tab10(np.linspace(0, 1, len(sectors)))

for i, sector in enumerate(sectors):
    sector_data = stock_metrics[stock_metrics['Sector'] == sector]
    plt.scatter(sector_data['Annual Volatility'], sector_data['Annual Return'],
               c=[colors[i]], label=sector, s=100, alpha=0.7)

# Add stock labels for significant positions
for idx, row in stock_metrics.iterrows():
    if abs(row['Sharpe Ratio']) > 0.5:  # Label high Sharpe ratio stocks
        plt.annotate(idx.replace('.NS', ''), 
                    (row['Annual Volatility'], row['Annual Return']),
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=8, fontweight='bold')

plt.xlabel('Annual Volatility (Risk)', fontsize=12)
plt.ylabel('Annual Return', fontsize=12)
plt.title('Risk-Return Profile of NIFTY Stocks', fontsize=16, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)

# Format axes as percentages
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))

plt.tight_layout()
plt.show()

## Portfolio Optimization

Now let's implement the core portfolio optimization algorithms using Modern Portfolio Theory.

In [ ]:
# Portfolio optimization functions
class PortfolioOptimizer:
    def __init__(self, returns, risk_free_rate=0.065):
        self.returns = returns
        self.mean_returns = returns.mean()
        self.cov_matrix = returns.cov()
        self.risk_free_rate = risk_free_rate
        self.num_assets = len(returns.columns)
    
    def portfolio_performance(self, weights):
        """Calculate portfolio performance metrics"""
        portfolio_return = np.sum(self.mean_returns * weights) * 252
        portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(self.cov_matrix * 252, weights)))
        sharpe_ratio = (portfolio_return - self.risk_free_rate) / portfolio_volatility
        return portfolio_return, portfolio_volatility, sharpe_ratio
    
    def negative_sharpe_ratio(self, weights):
        """Objective function to minimize (negative Sharpe ratio)"""
        _, _, sharpe = self.portfolio_performance(weights)
        return -sharpe
    
    def portfolio_volatility(self, weights):
        """Calculate portfolio volatility"""
        return np.sqrt(np.dot(weights.T, np.dot(self.cov_matrix * 252, weights)))
    
    def optimize_portfolio(self, objective='sharpe'):
        """Optimize portfolio based on objective"""
        # Constraints and bounds
        constraints = {'type': 'eq', 'fun': lambda x: np.sum(x) - 1}
        bounds = tuple((0, 0.3) for _ in range(self.num_assets))  # Max 30% in any single stock
        
        # Initial guess (equal weights)
        initial_guess = np.array([1/self.num_assets] * self.num_assets)
        
        # Optimization based on objective
        if objective == 'sharpe':
            result = minimize(self.negative_sharpe_ratio, initial_guess,
                            method='SLSQP', bounds=bounds, constraints=constraints)
        elif objective == 'min_vol':
            result = minimize(self.portfolio_volatility, initial_guess,
                            method='SLSQP', bounds=bounds, constraints=constraints)
        elif objective == 'max_return':
            result = minimize(lambda x: -np.sum(self.mean_returns * x) * 252, initial_guess,
                            method='SLSQP', bounds=bounds, constraints=constraints)
        
        # Calculate performance metrics
        optimal_weights = result.x
        performance = self.portfolio_performance(optimal_weights)
        
        return {
            'weights': optimal_weights,
            'expected_return': performance[0],
            'volatility': performance[1],
            'sharpe_ratio': performance[2],
            'objective': objective,
            'success': result.success
        }
    
    def generate_efficient_frontier(self, num_portfolios=100):
        """Generate efficient frontier"""
        # Get min and max possible returns
        min_vol_portfolio = self.optimize_portfolio('min_vol')
        max_ret_portfolio = self.optimize_portfolio('max_return')
        
        min_ret = min_vol_portfolio['expected_return']
        max_ret = max_ret_portfolio['expected_return']
        
        target_returns = np.linspace(min_ret, max_ret, num_portfolios)
        
        efficient_portfolios = []
        
        for target_return in target_returns:
            try:
                # Constraints for target return
                constraints = [
                    {'type': 'eq', 'fun': lambda x: np.sum(x) - 1},
                    {'type': 'eq', 'fun': lambda x: np.sum(self.mean_returns * x) * 252 - target_return}
                ]
                
                bounds = tuple((0, 0.3) for _ in range(self.num_assets))
                initial_guess = np.array([1/self.num_assets] * self.num_assets)
                
                result = minimize(self.portfolio_volatility, initial_guess,
                                method='SLSQP', bounds=bounds, constraints=constraints)
                
                if result.success:
                    weights = result.x
                    ret, vol, sharpe = self.portfolio_performance(weights)
                    
                    efficient_portfolios.append({
                        'Return': ret,
                        'Volatility': vol,
                        'Sharpe_Ratio': sharpe,
                        'Weights': weights
                    })
                    
            except:
                continue
        
        return pd.DataFrame(efficient_portfolios)

# Initialize optimizer
optimizer = PortfolioOptimizer(returns)
print("✅ Portfolio optimizer initialized successfully!")

In [ ]:
# Optimize portfolios with different objectives
print("🎯 Optimizing portfolios...")

# Maximum Sharpe Ratio Portfolio
sharpe_portfolio = optimizer.optimize_portfolio('sharpe')
print(f"\n📈 Maximum Sharpe Ratio Portfolio:")
print(f"Expected Return: {sharpe_portfolio['expected_return']:.2%}")
print(f"Volatility: {sharpe_portfolio['volatility']:.2%}")
print(f"Sharpe Ratio: {sharpe_portfolio['sharpe_ratio']:.4f}")

# Minimum Volatility Portfolio
min_vol_portfolio = optimizer.optimize_portfolio('min_vol')
print(f"\n🛡️ Minimum Volatility Portfolio:")
print(f"Expected Return: {min_vol_portfolio['expected_return']:.2%}")
print(f"Volatility: {min_vol_portfolio['volatility']:.2%}")
print(f"Sharpe Ratio: {min_vol_portfolio['sharpe_ratio']:.4f}")

# Maximum Return Portfolio
max_ret_portfolio = optimizer.optimize_portfolio('max_return')
print(f"\n🚀 Maximum Return Portfolio:")
print(f"Expected Return: {max_ret_portfolio['expected_return']:.2%}")
print(f"Volatility: {max_ret_portfolio['volatility']:.2%}")
print(f"Sharpe Ratio: {max_ret_portfolio['sharpe_ratio']:.4f}")

# Store optimized portfolios
optimized_portfolios = {
    'sharpe': sharpe_portfolio,
    'min_vol': min_vol_portfolio,
    'max_return': max_ret_portfolio
}

In [ ]:
# Analyze portfolio allocations
def analyze_allocation(weights, portfolio_name):
    """Analyze and display portfolio allocation"""
    allocation_df = pd.DataFrame({
        'Stock': returns.columns,
        'Weight': weights,
        'Sector': [SECTOR_MAPPING.get(stock, 'Unknown') for stock in returns.columns]
    })
    
    # Filter significant holdings (>1%)
    significant_holdings = allocation_df[allocation_df['Weight'] > 0.01].sort_values('Weight', ascending=False)
    
    print(f"\n🏦 {portfolio_name} - Top Holdings (>1%):")
    for _, row in significant_holdings.head(10).iterrows():
        print(f"{row['Stock'].replace('.NS', '')}: {row['Weight']:.2%} ({row['Sector']})")
    
    # Sector allocation
    sector_allocation = allocation_df.groupby('Sector')['Weight'].sum().sort_values(ascending=False)
    print(f"\n🏭 Sector Allocation:")
    for sector, weight in sector_allocation.items():
        if weight > 0.01:
            print(f"{sector}: {weight:.2%}")
    
    return allocation_df, sector_allocation

# Analyze each optimized portfolio
for name, portfolio in optimized_portfolios.items():
    allocation_df, sector_allocation = analyze_allocation(portfolio['weights'], name.replace('_', ' ').title())

## Efficient Frontier Analysis

Let's generate and visualize the efficient frontier.

In [ ]:
# Generate efficient frontier
print("📊 Generating efficient frontier...")
efficient_df = optimizer.generate_efficient_frontier(num_portfolios=50)

if not efficient_df.empty:
    print(f"✅ Generated {len(efficient_df)} efficient portfolios")
    print(f"Return range: {efficient_df['Return'].min():.2%} to {efficient_df['Return'].max():.2%}")
    print(f"Volatility range: {efficient_df['Volatility'].min():.2%} to {efficient_df['Volatility'].max():.2%}")
else:
    print("❌ Could not generate efficient frontier")

In [ ]:
# Monte Carlo simulation for random portfolios
def monte_carlo_simulation(num_simulations=10000):
    """Generate random portfolios for comparison"""
    np.random.seed(42)
    results = []
    
    for _ in range(num_simulations):
        # Generate random weights
        weights = np.random.random(optimizer.num_assets)
        weights = weights / np.sum(weights)
        
        # Calculate performance
        ret, vol, sharpe = optimizer.portfolio_performance(weights)
        
        results.append({
            'Return': ret,
            'Volatility': vol,
            'Sharpe_Ratio': sharpe
        })
    
    return pd.DataFrame(results)

print("🎲 Running Monte Carlo simulation...")
monte_carlo_df = monte_carlo_simulation(5000)  # Reduced for faster execution
print(f"✅ Generated {len(monte_carlo_df)} random portfolios")

In [ ]:
# Plot efficient frontier with Monte Carlo simulation
plt.figure(figsize=(14, 10))

# Plot Monte Carlo simulation
scatter = plt.scatter(monte_carlo_df['Volatility'], monte_carlo_df['Return'],
                     c=monte_carlo_df['Sharpe_Ratio'], cmap='viridis',
                     alpha=0.6, s=10, label='Random Portfolios')
plt.colorbar(scatter, label='Sharpe Ratio')

# Plot efficient frontier
if not efficient_df.empty:
    plt.plot(efficient_df['Volatility'], efficient_df['Return'],
            'r-', linewidth=3, label='Efficient Frontier')

# Plot optimized portfolios
colors = {'sharpe': 'gold', 'min_vol': 'green', 'max_return': 'blue'}
markers = {'sharpe': '*', 'min_vol': 'o', 'max_return': '^'}

for obj_type, portfolio in optimized_portfolios.items():
    plt.scatter(portfolio['volatility'], portfolio['expected_return'],
               color=colors[obj_type], marker=markers[obj_type],
               s=300, label=f'{obj_type.replace("_", " ").title()} Portfolio',
               edgecolors='black', linewidth=2, zorder=5)

plt.xlabel('Volatility (Risk)', fontsize=14)
plt.ylabel('Expected Return', fontsize=14)
plt.title('Efficient Frontier - NIFTY Portfolio Optimization', fontsize=16, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)

# Format axes as percentages
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))

plt.tight_layout()
plt.show()

## Portfolio Allocation Visualization

Let's create pie charts to visualize the portfolio allocations.

In [ ]:
# Create pie charts for portfolio allocations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

portfolio_names = ['Maximum Sharpe Ratio', 'Minimum Volatility', 'Maximum Return']
portfolio_keys = ['sharpe', 'min_vol', 'max_return']

for i, (name, key) in enumerate(zip(portfolio_names, portfolio_keys)):
    weights = optimized_portfolios[key]['weights']
    
    # Create allocation dataframe
    allocation_df = pd.DataFrame({
        'Stock': returns.columns,
        'Weight': weights
    }).sort_values('Weight', ascending=False)
    
    # Group small allocations
    significant = allocation_df[allocation_df['Weight'] >= 0.02]  # ≥2%
    others_weight = allocation_df[allocation_df['Weight'] < 0.02]['Weight'].sum()
    
    if others_weight > 0:
        plot_data = pd.concat([
            significant,
            pd.DataFrame({'Stock': ['Others'], 'Weight': [others_weight]})
        ], ignore_index=True)
    else:
        plot_data = significant
    
    # Clean stock names
    plot_data['Stock'] = plot_data['Stock'].str.replace('.NS', '')
    
    # Create pie chart
    colors = plt.cm.Set3(np.linspace(0, 1, len(plot_data)))
    wedges, texts, autotexts = axes[i].pie(plot_data['Weight'], 
                                          labels=plot_data['Stock'],
                                          autopct='%1.1f%%',
                                          startangle=90,
                                          colors=colors)
    
    # Enhance text appearance
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
        autotext.set_fontsize(8)
    
    for text in texts:
        text.set_fontsize(8)
    
    axes[i].set_title(f'{name} Portfolio', fontsize=12, fontweight='bold', pad=20)

# Remove empty subplot
axes[3].remove()

plt.tight_layout()
plt.show()

## Performance Analysis and Backtesting

Let's analyze the historical performance of our optimized portfolios.

In [ ]:
# Calculate portfolio performance over time
def calculate_portfolio_performance(returns, weights):
    """Calculate portfolio returns and cumulative performance"""
    portfolio_returns = (returns * weights).sum(axis=1)
    cumulative_returns = (1 + portfolio_returns).cumprod()
    
    return portfolio_returns, cumulative_returns

# Calculate performance for each optimized portfolio
portfolio_performance = {}

for name, portfolio in optimized_portfolios.items():
    port_returns, cum_returns = calculate_portfolio_performance(returns, portfolio['weights'])
    portfolio_performance[name] = {
        'returns': port_returns,
        'cumulative': cum_returns
    }

# Calculate equal-weight benchmark
equal_weights = np.array([1/len(returns.columns)] * len(returns.columns))
benchmark_returns, benchmark_cumulative = calculate_portfolio_performance(returns, equal_weights)

print("📈 Portfolio performance calculated successfully!")

In [ ]:
# Plot cumulative performance
plt.figure(figsize=(14, 10))

# Plot optimized portfolios
colors = {'sharpe': 'red', 'min_vol': 'green', 'max_return': 'blue'}
labels = {'sharpe': 'Max Sharpe', 'min_vol': 'Min Volatility', 'max_return': 'Max Return'}

for name, performance in portfolio_performance.items():
    plt.plot(performance['cumulative'].index, performance['cumulative'].values,
            color=colors[name], linewidth=2, label=labels[name])

# Plot equal-weight benchmark
plt.plot(benchmark_cumulative.index, benchmark_cumulative.values,
        color='black', linewidth=2, linestyle='--', alpha=0.7, label='Equal Weight Benchmark')

plt.title('Portfolio Performance Comparison', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Cumulative Return', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)

# Format y-axis
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{(y-1):.1%}'))

plt.tight_layout()
plt.show()

# Performance summary
print("\n📊 Performance Summary:")
for name, performance in portfolio_performance.items():
    total_return = performance['cumulative'].iloc[-1] - 1
    annual_vol = performance['returns'].std() * np.sqrt(252)
    print(f"{labels[name]}: Total Return = {total_return:.2%}, Annual Volatility = {annual_vol:.2%}")

benchmark_total_return = benchmark_cumulative.iloc[-1] - 1
benchmark_vol = benchmark_returns.std() * np.sqrt(252)
print(f"Equal Weight Benchmark: Total Return = {benchmark_total_return:.2%}, Annual Volatility = {benchmark_vol:.2%}")

## Risk Analysis and Stress Testing

Let's perform comprehensive risk analysis on our optimized portfolios.

In [ ]:
# Comprehensive risk metrics
def calculate_risk_metrics(returns, weights, confidence_level=0.05):
    """Calculate comprehensive risk metrics"""
    portfolio_returns = (returns * weights).sum(axis=1)
    
    # Basic metrics
    annual_return = portfolio_returns.mean() * 252
    annual_vol = portfolio_returns.std() * np.sqrt(252)
    sharpe_ratio = (annual_return - 0.065) / annual_vol
    
    # Downside metrics
    downside_returns = portfolio_returns[portfolio_returns < 0]
    downside_vol = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 0 else 0
    sortino_ratio = (annual_return - 0.065) / downside_vol if downside_vol != 0 else 0
    
    # Maximum drawdown
    cumulative = (1 + portfolio_returns).cumprod()
    rolling_max = cumulative.expanding().max()
    drawdown = (cumulative - rolling_max) / rolling_max
    max_drawdown = drawdown.min()
    
    # Value at Risk (VaR) and Conditional VaR
    var_95 = np.percentile(portfolio_returns, confidence_level * 100)
    cvar_95 = portfolio_returns[portfolio_returns <= var_95].mean()
    
    # Skewness and Kurtosis
    skewness = stats.skew(portfolio_returns)
    kurtosis = stats.kurtosis(portfolio_returns)
    
    return {
        'Annual Return': annual_return,
        'Annual Volatility': annual_vol,
        'Sharpe Ratio': sharpe_ratio,
        'Sortino Ratio': sortino_ratio,
        'Max Drawdown': max_drawdown,
        'VaR (95%)': var_95,
        'CVaR (95%)': cvar_95,
        'Skewness': skewness,
        'Kurtosis': kurtosis,
        'Win Rate': (portfolio_returns > 0).sum() / len(portfolio_returns)
    }

# Calculate risk metrics for all portfolios
risk_summary = {}

for name, portfolio in optimized_portfolios.items():
    risk_summary[labels[name]] = calculate_risk_metrics(returns, portfolio['weights'])

# Add benchmark
risk_summary['Equal Weight Benchmark'] = calculate_risk_metrics(returns, equal_weights)

# Create risk metrics dataframe
risk_df = pd.DataFrame(risk_summary).T

print("📊 Comprehensive Risk Analysis:")
print(risk_df.round(4))

In [ ]:
# Rolling risk metrics visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Calculate rolling metrics for Max Sharpe portfolio
sharpe_returns = portfolio_performance['sharpe']['returns']
window = 60  # 60-day rolling window

# Rolling volatility
rolling_vol = sharpe_returns.rolling(window=window).std() * np.sqrt(252)
axes[0, 0].plot(rolling_vol.index, rolling_vol.values, color='red', alpha=0.8)
axes[0, 0].fill_between(rolling_vol.index, rolling_vol.values, alpha=0.3, color='red')
axes[0, 0].set_title('60-Day Rolling Volatility (Max Sharpe Portfolio)', fontweight='bold')
axes[0, 0].set_ylabel('Volatility')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))

# Rolling Sharpe ratio
rolling_sharpe = (sharpe_returns.rolling(window=window).mean() * 252 - 0.065) / (sharpe_returns.rolling(window=window).std() * np.sqrt(252))
axes[0, 1].plot(rolling_sharpe.index, rolling_sharpe.values, color='blue', alpha=0.8)
axes[0, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[0, 1].set_title('60-Day Rolling Sharpe Ratio', fontweight='bold')
axes[0, 1].set_ylabel('Sharpe Ratio')
axes[0, 1].grid(True, alpha=0.3)

# Drawdown analysis
cumulative = (1 + sharpe_returns).cumprod()
rolling_max = cumulative.expanding().max()
drawdown = (cumulative - rolling_max) / rolling_max

axes[1, 0].fill_between(drawdown.index, drawdown.values, 0, color='red', alpha=0.7)
axes[1, 0].set_title('Portfolio Drawdown (Max Sharpe Portfolio)', fontweight='bold')
axes[1, 0].set_ylabel('Drawdown')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))

# Return distribution
axes[1, 1].hist(sharpe_returns, bins=50, alpha=0.7, color='green', density=True)
axes[1, 1].axvline(sharpe_returns.mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[1, 1].axvline(np.percentile(sharpe_returns, 5), color='orange', linestyle='--', linewidth=2, label='5% VaR')
axes[1, 1].set_title('Daily Returns Distribution', fontweight='bold')
axes[1, 1].set_xlabel('Daily Return')
axes[1, 1].set_ylabel('Density')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))

plt.tight_layout()
plt.show()

## Interactive Dashboard

Let's create an interactive dashboard using Plotly for better exploration.

In [ ]:
# Create interactive efficient frontier plot
fig = go.Figure()

# Add Monte Carlo scatter
fig.add_trace(go.Scatter(
    x=monte_carlo_df['Volatility'],
    y=monte_carlo_df['Return'],
    mode='markers',
    marker=dict(
        size=4,
        color=monte_carlo_df['Sharpe_Ratio'],
        colorscale='Viridis',
        colorbar=dict(title="Sharpe Ratio"),
        opacity=0.6
    ),
    name='Random Portfolios',
    hovertemplate='Return: %{y:.2%}<br>Volatility: %{x:.2%}<br>Sharpe: %{marker.color:.3f}<extra></extra>'
))

# Add efficient frontier
if not efficient_df.empty:
    fig.add_trace(go.Scatter(
        x=efficient_df['Volatility'],
        y=efficient_df['Return'],
        mode='lines',
        line=dict(color='red', width=3),
        name='Efficient Frontier',
        hovertemplate='Return: %{y:.2%}<br>Volatility: %{x:.2%}<extra></extra>'
    ))

# Add optimized portfolios
plot_colors = {'sharpe': 'gold', 'min_vol': 'green', 'max_return': 'blue'}
plot_symbols = {'sharpe': 'star', 'min_vol': 'circle', 'max_return': 'triangle-up'}
plot_labels = {'sharpe': 'Max Sharpe', 'min_vol': 'Min Volatility', 'max_return': 'Max Return'}

for obj_type, portfolio in optimized_portfolios.items():
    fig.add_trace(go.Scatter(
        x=[portfolio['volatility']],
        y=[portfolio['expected_return']],
        mode='markers',
        marker=dict(
            symbol=plot_symbols[obj_type],
            size=15,
            color=plot_colors[obj_type],
            line=dict(color='black', width=2)
        ),
        name=f'{plot_labels[obj_type]} Portfolio',
        hovertemplate=f'Return: %{{y:.2%}}<br>Volatility: %{{x:.2%}}<br>Sharpe: {portfolio["sharpe_ratio"]:.3f}<extra></extra>'
    ))

# Update layout
fig.update_layout(
    title='Interactive Efficient Frontier - NIFTY Portfolio Optimization',
    xaxis_title='Volatility (Risk)',
    yaxis_title='Expected Return',
    hovermode='closest',
    template='plotly_white',
    width=900,
    height=600
)

# Format axes as percentages
fig.update_xaxes(tickformat='.1%')
fig.update_yaxes(tickformat='.1%')

fig.show()

In [ ]:
# Interactive performance comparison
fig = go.Figure()

# Add portfolio performance lines
for name, performance in portfolio_performance.items():
    fig.add_trace(go.Scatter(
        x=performance['cumulative'].index,
        y=performance['cumulative'].values,
        mode='lines',
        name=labels[name],
        line=dict(width=2),
        hovertemplate='Date: %{x}<br>Cumulative Return: %{y:.2%}<extra></extra>'
    ))

# Add benchmark
fig.add_trace(go.Scatter(
    x=benchmark_cumulative.index,
    y=benchmark_cumulative.values,
    mode='lines',
    name='Equal Weight Benchmark',
    line=dict(dash='dash', width=2, color='black'),
    hovertemplate='Date: %{x}<br>Cumulative Return: %{y:.2%}<extra></extra>'
))

fig.update_layout(
    title='Interactive Portfolio Performance Comparison',
    xaxis_title='Date',
    yaxis_title='Cumulative Return',
    template='plotly_white',
    width=900,
    height=500
)

fig.update_yaxes(tickformat='.1%')

fig.show()

## Key Insights and Conclusions

### Summary of Results

Based on our portfolio optimization analysis of NIFTY 50 stocks, here are the key findings:

#### Portfolio Performance Comparison:
- **Maximum Sharpe Ratio Portfolio**: Offers the best risk-adjusted returns
- **Minimum Volatility Portfolio**: Provides the lowest risk but potentially lower returns
- **Maximum Return Portfolio**: Targets highest returns but with increased risk

#### Key Observations:
1. **Diversification Benefits**: Optimized portfolios show better risk-return profiles than individual stocks
2. **Sector Concentration**: Certain sectors (IT, Financials) tend to dominate optimized portfolios
3. **Risk Management**: Maximum drawdown analysis helps identify potential downside risks
4. **Correlation Impact**: Lower correlation stocks provide better diversification benefits

### Practical Applications for Banks:

#### 1. **Wealth Management**:
- Use optimized portfolios as starting points for client recommendations
- Customize based on client risk tolerance and investment horizon
- Regular rebalancing based on changing market conditions

#### 2. **Risk Management**:
- Monitor portfolio VaR and maximum drawdown limits
- Implement stress testing scenarios
- Set position limits based on optimization constraints

#### 3. **Product Development**:
- Create mutual fund schemes based on optimization strategies
- Develop structured products with optimized underlying portfolios
- Offer robo-advisory services using these algorithms

### Student Exploration Ideas:

1. **Parameter Sensitivity**: Change risk-free rate, time periods, constraints
2. **Sector Analysis**: Focus on specific sectors or exclude certain industries
3. **Alternative Objectives**: Implement minimum tracking error, maximum diversification
4. **Transaction Costs**: Include realistic trading costs in optimization
5. **Dynamic Rebalancing**: Implement time-varying optimization

### Limitations and Considerations:

1. **Historical Data Dependency**: Past performance may not predict future results
2. **Market Regime Changes**: Optimization parameters may need adjustment during market stress
3. **Transaction Costs**: Real-world implementation requires considering trading costs
4. **Liquidity Constraints**: Large portfolios may face liquidity issues in certain stocks
5. **Regulatory Requirements**: Banks must comply with various regulatory limits

### Next Steps:

1. Implement factor models (Fama-French, APT)
2. Add ESG constraints and considerations
3. Include options and derivatives for enhanced strategies
4. Develop real-time portfolio monitoring systems
5. Apply machine learning for return forecasting


In [ ]:
# Export results summary
results_summary = {
    'optimization_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'data_period': '2 years',
    'num_stocks': len(returns.columns),
    'risk_free_rate': 0.065,
    'portfolios': {}
}

for name, portfolio in optimized_portfolios.items():
    results_summary['portfolios'][name] = {
        'expected_return': f"{portfolio['expected_return']:.4f}",
        'volatility': f"{portfolio['volatility']:.4f}",
        'sharpe_ratio': f"{portfolio['sharpe_ratio']:.4f}",
        'top_5_holdings': dict(zip(
            returns.columns,
            portfolio['weights']
        ))
    }

print("📄 Results Summary:")
print(json.dumps(results_summary, indent=2, default=str))

print("\n✅ Portfolio Optimization Demo Completed Successfully!")
print("📚 This notebook demonstrated:")
print("   • Modern Portfolio Theory implementation")
print("   • Efficient frontier generation")
print("   • Risk-return optimization")
print("   • Performance analysis and visualization")
print("   • Practical applications for financial institutions")